# BƯỚC 3: KỸ THUẬT ĐẶC TRƯNG (FEATURE ENGINEERING)

**Mục tiêu:** Tạo ra các biến số mới (Features) từ bộ dữ liệu đã được làm sạch và ghép nối ở Bước 2, giúp mô hình Học máy có thêm "giác quan" để bắt mạch thị trường.

**Các kỹ thuật áp dụng theo barem:**
1. **Lag Features (Đặc trưng trễ):** Khai thác tính chất chuỗi thời gian (Giá ngày hôm qua, tuần trước).
2. **Technical Indicators (Chỉ báo kỹ thuật):** Áp dụng công thức tài chính để tính toán Đường trung bình động (SMA, EMA), Chỉ số sức mạnh tương đối (RSI), và MACD.
3. **Target Variable (Biến mục tiêu):** Thiết lập nhãn `Target_Class` (1 = Giá Tăng, 0 = Giá Giảm/Đi ngang) phục vụ cho bài toán Phân loại (Classification).

## 3.1. Nạp dữ liệu đã Tiền xử lý (Từ Bước 2)
Nạp các file `_scaled.csv` đã được lấp đầy khuyết thiếu và ghép sẵn VNINDEX cùng tỷ giá USD/VND.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

print("--- 3.1: Load dữ liệu đã tiền xử lý ---")

data_frames = {}
target_tickers = ['VNM', 'FPT', 'VIC']
processed_dir = '../data/processed'

for ticker in target_tickers:
    file_path = os.path.join(processed_dir, f"{ticker}_scaled.csv")
    
    if not os.path.exists(file_path):
        print(f"[LỖI] Không tìm thấy file: {file_path}. Hãy chạy lại Bước 2 & 3!")
        continue
        
    df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
    data_frames[ticker] = df
    
    print(f"-> Đã nạp {ticker} ({df.shape[0]} dòng, {df.shape[1]} cột)")

## 3.2. Tính toán Chỉ báo (Indicators) và Đặc trưng trễ (Lag Features)
Chúng ta sẽ định nghĩa một hàm tự động tính toán toàn bộ các chỉ báo kỹ thuật cơ bản dựa trên cột giá đóng cửa (`Close`).

In [ ]:
print("--- 3.2: Feature Engineering (Tạo chỉ báo & Đặc trưng trễ) ---")

def create_features(df):
    df_feat = df.copy()
    
    #Xóa khoảng trắng thừa ở tên cột
    df_feat.columns = df_feat.columns.str.strip()
    
    #Lag features
    for i in [1, 2, 3]:
        df_feat[f'Close_Lag_{i}'] = df_feat['Close'].shift(i)
        df_feat[f'Volume_Lag_{i}'] = df_feat['Volume'].shift(i)
        
    #Moving Averages (SMA, EMA)
    df_feat['SMA_10'] = df_feat['Close'].rolling(window=10).mean()
    df_feat['SMA_30'] = df_feat['Close'].rolling(window=30).mean()
    df_feat['EMA_10'] = df_feat['Close'].ewm(span=10, adjust=False).mean()
    
    #RSI
    delta = df_feat['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
    rs = gain / loss
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))
    df_feat['RSI_14'] = df_feat['RSI_14'] / 100.0 
    
    #MACD 
    ema_12 = df_feat['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_feat['Close'].ewm(span=26, adjust=False).mean()
    df_feat['MACD'] = ema_12 - ema_26
    df_feat['MACD_Signal'] = df_feat['MACD'].ewm(span=9, adjust=False).mean()

    #Bollinger Bands
    bb_mid = df_feat['Close'].rolling(window=20).mean()
    bb_std = df_feat['Close'].rolling(window=20).std()
    
    df_feat['BB_Mid'] = bb_mid
    df_feat['BB_Upper'] = bb_mid + (bb_std * 2)
    df_feat['BB_Lower'] = bb_mid - (bb_std * 2)
    
    #Đặc trưng thời gian & Tỷ suất sinh lời
    df_feat['Day_Of_Week'] = df_feat.index.dayofweek 
    df_feat['Return_1D'] = df_feat['Close'].pct_change()
    df_feat['Volume_Change'] = df_feat['Volume'].pct_change()
    df_feat['Price_to_SMA30'] = df_feat['Close'] / df_feat['SMA_30']
    
    #Xử lý các cột tham chiếu
    if 'VNINDEX_Close' in df_feat.columns:
        df_feat['VNINDEX_Return'] = df_feat['VNINDEX_Close'].pct_change()
        
    if 'USDVND_Close' in df_feat.columns:
        df_feat['USDVND_Change'] = df_feat['USDVND_Close'].pct_change()

    #Chuyển vô cực thành NaN để dễ xử lý về sau
    df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    return df_feat


#Áp dụng hàm tạo đặc trưng
for ticker, df in data_frames.items():
    data_frames[ticker] = create_features(df)
    print(f"-> Đã tạo features cho {ticker} (Tổng: {data_frames[ticker].shape[1]} cột)")

## 3.3. Thiết lập Biến mục tiêu (Target) và Xuất file

Biến mục tiêu cho bài toán **Phân loại (Classification)**:
- `Target_Class` = 1: Nếu giá đóng cửa ngày mai CAO HƠN giá đóng cửa hôm nay.
- `Target_Class` = 0: Nếu giá đóng cửa ngày mai THẤP HƠN hoặc BẰNG hôm nay.

In [ ]:
print("--- 3.3: Thiết lập Target (Nhãn dự đoán) & Clean Data ---")

processed_dir = '../data/processed'

for ticker, df in data_frames.items():
    #Tạo nhãn Classification
    df['Target_Class'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    #Xóa các dòng chứa NaN
    df_clean = df.dropna()
    data_frames[ticker] = df_clean
    
    #Lưu data hoàn chỉnh sẵn sàng cho Model
    final_file = os.path.join(processed_dir, f"{ticker}_features.csv")
    df_clean.to_csv(final_file)
    
    print(f"-> Hoàn thiện {ticker}: {df_clean.shape[0]} dòng, {df_clean.shape[1]} cột. Đã lưu file.")

In [ ]:
print("--- 3.4: Trực quan hóa Ma trận Tương quan (Correlation Heatmap) ---")

output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

#Khai báo list cột cần xem xét ở ngoài vòng lặp
cols_to_plot = ['Close', 'Volume', 'SMA_10', 'RSI_14', 'MACD', 'Target_Class']

for ticker, df in data_frames.items():
    #Lọc nhanh các cột thực tế đang có trong DataFrame
    valid_cols = [col for col in cols_to_plot if col in df.columns]
    
    #Thoát nếu không có cột nào hợp lệ
    if not valid_cols:
        print(f"[{ticker}] Không đủ dữ liệu để vẽ Heatmap.")
        continue
        
    corr_matrix = df[valid_cols].corr()

    #Cấu hình và vẽ Heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', fmt=".2f", linewidths=0.5)
    
    plt.title(f'Correlation Matrix - {ticker}')
    plt.tight_layout()
    
    #Lưu file
    save_path = os.path.join(output_dir, f'03_correlation_{ticker}.png')
    plt.savefig(save_path, dpi=300)
    plt.show()
    
    print(f"-> Đã xuất Heatmap cho {ticker}")